# InfraVision: train the ThermoVision colorizer on LLVIP (Kaggle)

Thermal → RGB U-Net (ResNet34 encoder), trained on 12,025 LLVIP pairs and scored on LLVIP test.
The script prints the time per epoch and an estimate of the time left. Expect a few hours on one T4.

**One-time setup**
1. Kaggle account → **Settings → Phone verification** (needed for GPU and internet).
2. **Datasets → New Dataset** → upload `llvip_colorizer.zip` (private). Any name is fine.
3. Here: **Add Input** → that dataset. **Session options**: Accelerator **GPU T4** (or P100), **Internet on**.
4. **Save Version → Save & Run All (Commit)**. It runs in the background; you can close the browser.
5. When it finishes: **Output** tab → download `colorizer_llvip.zip`. Copy `thermovision_best.pth` from it to
   `InfraVision/checkpoints/colorizer/thermovision_llvip_v1.pth` on the laptop.

**If the run stops early** (12 h limit, error): add this notebook's previous version output as an input and run again.
The cell below finds its `last.pth` and training continues from the last finished epoch.


In [ ]:
import shutil, subprocess
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
REPO, OUT = WORK / "InfraVision", WORK / "colorizer"

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", "feature/rev1",
                    "https://github.com/elzasabu/InfraVision.git", str(REPO)], check=True)

# The unzipped dataset, wherever Kaggle mounted it
data = [p.parent.parent for p in INPUT.rglob("ir") if p.parent.name == "train" and (p.parent / "rgb").is_dir()]
assert data, "LLVIP zip not found: Add Input -> your llvip_colorizer dataset"
DATA = data[0]

# Continue an earlier run if its output was added as an input
OUT.mkdir(parents=True, exist_ok=True)
previous = list(INPUT.rglob("last.pth"))
if previous and not (OUT / "last.pth").exists():
    shutil.copy(previous[0], OUT / "last.pth")
    print("continuing from", previous[0])

import torch
print("data:", DATA, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU")


In [ ]:
# A failed "!" command does not stop the notebook, so check the exit code.
p = subprocess.Popen(["python", "-m", "infravision.train.thermovision", '--data', str(DATA), '--out', str(OUT), '--amp', '--resume', '--workers', '4', '--epochs', '30', '--batch', '16'], cwd=REPO,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:   # live log in the cell and in Kaggle's Logs tab
    print(line, end="", flush=True)
assert p.wait() == 0, "training failed: see the error above"

In [ ]:
import json
from IPython.display import Image as Show, display

m = json.loads((OUT / "metrics.json").read_text())
print("best epoch:", json.dumps(m["best"], indent=2))
for h in m["history"]:
    print(f"epoch {h['epoch']:3d}  PSNR {h['val_psnr']:.2f}  SSIM {h['val_ssim']:.3f}  saturation {h['val_saturation']:.3f}")
display(Show(str(sorted(OUT.glob("val_samples_epoch*.png"))[-1])))   # rows: thermal | prediction | real RGB


## Package the results (best weights, metrics, sample images; `last.pth` stays in the Output for resuming)

In [ ]:
pack = WORK / "colorizer_pack"
pack.mkdir(exist_ok=True)
for f in OUT.iterdir():
    if f.name != "last.pth":
        shutil.copy(f, pack / f.name)
print("Saved:", shutil.make_archive(str(WORK / "colorizer_llvip"), "zip", pack))
